In [ ]:
!pip install rectools[lightfm,torch] lightning-fabric

In [2]:
from google.colab import drive
drive.mount("/gdrive", force_remount=True)

Mounted at /gdrive


In [3]:
!ls -l /gdrive/MyDrive/RecSys-Course/data/

total 9073
drwx------ 2 root root    4096 Mar  7 07:39 2026-03-07-8x-10k-full-random
drwx------ 2 root root    4096 Mar  7 20:59 2026-03-07-8x-1k-full-random
drwx------ 2 root root    4096 Mar  8 19:15 2026-03-07-8x-50k-full-random
-rw------- 1 root root 9277528 Aug 26  2025 tracks.json


In [83]:
import os
import glob
import json

import pandas as pd
import numpy as np

from lightfm import LightFM

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import PopularModel, LightFMWrapperModel, SASRecModel
from rectools.metrics import NDCG, HitRate
from rectools.visuals.visual_app import ItemToItemVisualApp

from lightning_fabric import seed_everything

In [5]:
# Enable deterministic behaviour with CUDA >= 10.2
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE, workers=True)

DATA_DIR = "/gdrive/MyDrive/RecSys-Course/data/"

INFO:lightning_fabric.utilities.seed:Seed set to 42


# Prepare dataset

In [6]:
train_data = pd.concat([
    pd.read_json(data_path, lines=True)[["user", "track", "timestamp", "time"]]
    for data_path
    in glob.glob(DATA_DIR + "2026-03-07-8x-50k-full-random/*/data.json")
])

# Use rectools column names
train_interactions = train_data[
    train_data["time"] > 0.7
].copy().rename(columns={
    "user": Columns.User,
    "track": Columns.Item,
    "timestamp": Columns.Datetime,
    "time": Columns.Weight
})

train_interactions.head(3)

,user_id,item_id,datetime,weight
0,3567,1218,2026-03-08 18:36:38.928,1.00
2,1707,10167,2026-03-08 18:36:38.938,0.88
4,3812,9194,2026-03-08 18:36:38.959,1.00


In [7]:
test_data = pd.concat([
    pd.read_json(data_path, lines=True)[["user", "track", "timestamp", "time"]]
    for data_path
    in glob.glob(DATA_DIR + "2026-03-07-8x-10k-full-random/*/data.json")
])

# Use rectools column names
test_interactions = test_data[
    (test_data["time"] > 0.7)
    & (test_data["user"].isin(train_interactions[Columns.User]))
    & (test_data["track"].isin(train_interactions[Columns.Item]))
].copy().rename(columns={
    "user": Columns.User,
    "track": Columns.Item,
    "timestamp": Columns.Datetime,
    "time": Columns.Weight
})

test_interactions.head(3)

,user_id,item_id,datetime,weight
0,3434,131,2026-03-07 07:19:28.488,1.00
5,5311,9464,2026-03-07 07:19:28.522,0.84
7,7105,2189,2026-03-07 07:19:28.534,0.83


In [66]:
tracks = pd.read_json(DATA_DIR + "tracks.json", lines=True).drop_duplicates(subset=["track"]).rename(columns={"track": Columns.Item})
tracks.head(1)

items = tracks.loc[tracks[Columns.Item].isin(train_interactions[Columns.Item])].copy()

artist_feature = items[[Columns.Item, "artist"]].copy()
artist_feature.columns = ["id", "value"]
artist_feature["feature"] = "artist"

genre_feature = items[[Columns.Item, "genres"]].explode("genres").copy()
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"

item_features = pd.concat((artist_feature, genre_feature))
item_features.head(3)

,id,value,feature
0,0,ABBA,artist
1,1,ABBA,artist
2,2,ABBA,artist


In [9]:
train_dataset = Dataset.construct(
    interactions_df=train_interactions,
    item_features_df=item_features,
    cat_item_features=["genre", "artist"],
)

test_dataset = Dataset.construct(
    interactions_df=test_interactions,
    item_features_df=item_features,
    cat_item_features=["genre", "artist"],
)

## Нейросетевой генератор кандидатов

#### Проблема:
Допустим, мы обучаем модель, которая ранжирует кандидатов. Отскорить всех кандидатов в real-time мы сможем за O(N). Это не подходит, если кандидатов очень много. Поэтому будем использовать приближенный поиск соседей (LSH, FAISS etc). При этом соседей мы будем находить по расстоянию между вектором юзера и векторами кандиадтов.

### Решение:
Давайте обучим отдельную модель -- отборщика кандидатов -- которая будет качественно искать ближайших соседей. Для этого смоделируем релевантность с помощью любой метрики, отражающей близость векторов.



In [10]:
popular = PopularModel()
popular.fit(train_dataset)
popular_recs = popular.recommend(
    users=test_interactions[Columns.User].unique(),
    dataset=train_dataset,
    k=10,
    filter_viewed=True,
)

In [11]:
%%time
lfm = LightFMWrapperModel(
    LightFM(no_components=64, loss="bpr", random_state=RANDOM_STATE),
    epochs=100
)
lfm.fit(train_dataset)
lfm_recs = lfm.recommend(
    users=test_interactions[Columns.User].unique(),
    dataset=train_dataset,
    k=10,
    filter_viewed=True,
)

CPU times: user 6min 57s, sys: 392 ms, total: 6min 58s
Wall time: 7min


In [56]:
sasrec = SASRecModel(
    session_max_len=100,
    loss="softmax",
    n_factors=64,
    n_blocks=1,
    n_heads=4,
    dropout_rate=0.2,
    lr=0.001,
    batch_size=128,
    epochs=50,
    verbose=1,
    deterministic=True,
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [57]:
%%time
sasrec.fit(train_dataset)

/usr/local/lib/python3.12/dist-packages/rectools/dataset/identifiers.py:60: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unq_values = pd.unique(values)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='item_net_block_types', input_value=('rectools.models.nn.item...net.CatFeaturesItemNet'), input_type=tuple])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type                     | Params | Mode 
--

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


CPU times: user 9min 57s, sys: 1.16 s, total: 9min 58s
Wall time: 10min 2s


In [58]:
sasrec_recs = sasrec.recommend(
    users=test_interactions[Columns.User].unique(),
    dataset=train_dataset,
    k=10,
    filter_viewed=True,
)

In [59]:
metrics = {
  "ndcg": NDCG(k=10, log_base=3),
  "hit_rate": HitRate(k=10),
}

recs = {
    "popular": popular_recs,
    "lightfm": lfm_recs,
    "sasrec": sasrec_recs,
}

results = []
for metric_name, metric in metrics.items():
  for model, model_recs in recs.items():
    results.append({
        "model": model,
        "metric": metric_name,
        "value": metric.calc(model_recs, interactions=test_interactions),
    })

pd.DataFrame(results).pivot(index='model', columns=['metric'], values='value').sort_values('ndcg')

metric,hit_rate,ndcg
model,,
popular,0.034010,0.003831
lightfm,0.253976,0.040200
sasrec,0.320196,0.054560


In [78]:
item_ids = train_interactions.groupby(Columns.Item).size().sort_values(ascending=False).index.to_list()[:15000]

lfm_i2i = lfm.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=10,
    filter_itself=True,
    items_to_recommend=None,
)

sasrec_i2i = sasrec.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=10,
    filter_itself=True,
    items_to_recommend=None,
)

In [81]:
ItemToItemVisualApp.construct(
    reco={"sasrec": sasrec_i2i, "lightfm": lfm_i2i},
    item_data=items[[Columns.Item, "title", "artist", "genres", "artist_country"]],
    n_random_items=6
)

In [86]:
def i2i_to_json(i2i_df, file_name):
    with open(file_name, "w") as json_file:
        for target_item_id, group in i2i_df.groupby('target_item_id'):
            recommendations = group.sort_values('rank')['item_id'].tolist()
            json_file.write(json.dumps({'item_id': target_item_id, 'recommendations': recommendations}) + "\n")


i2i_to_json(sasrec_i2i, DATA_DIR + "sasrec.jsonl")
i2i_to_json(lfm_i2i, DATA_DIR + "lightfm.jsonl")

In [89]:
!head /gdrive/MyDrive/RecSys-Course/data/sasrec.jsonl

{"item_id": 0, "recommendations": [2, 7133, 6, 7, 1, 7134, 9, 7104, 8, 3]}
{"item_id": 1, "recommendations": [2, 7, 3, 9, 8, 0, 7134, 13, 6, 3417]}
{"item_id": 2, "recommendations": [1, 7, 8, 3, 0, 9, 7133, 5, 6, 13]}
{"item_id": 3, "recommendations": [1, 9, 2, 8, 7, 5, 3417, 3423, 13, 7104]}
{"item_id": 4, "recommendations": [5, 14408, 6942, 14407, 15986, 6, 0, 7, 15990, 14474]}
{"item_id": 5, "recommendations": [4, 7114, 8, 3, 2, 14408, 7, 6942, 1, 7104]}
{"item_id": 6, "recommendations": [0, 7, 7133, 4, 2, 1, 7134, 15913, 15986, 610]}
{"item_id": 7, "recommendations": [2, 1, 8, 9, 0, 6, 3, 7134, 7133, 5]}
{"item_id": 8, "recommendations": [2, 1, 7, 3, 9, 5, 7104, 0, 13, 3406]}
{"item_id": 9, "recommendations": [1, 3, 7, 2, 8, 3417, 3423, 13, 0, 5]}
